<a href="https://colab.research.google.com/github/fcovarrubiasp-debug/CIENCIA-DE-DATO-/blob/main/7_S2_Notebook_Laboratorio_CDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 2 — Calidad y preparación de tu conjunto de datos
**Ciencia de Datos · Semana 2 · Advance UCSC**

| | |
|---|---|
| **Resultado de aprendizaje** | RA1 (cierre) · RA3 (transversal) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Insumo** | El conjunto de datos que elegiste en la semana 1 |
| **Producto** | Conjunto limpio + bitácora completada + avance versionado |
| **Estudiante** | _(Fernando Copvarrubias Pizarro)_ |

---

Este laboratorio **no trae los datos**: los traes tú. Todo lo que hagas aquí es el cuerpo de la **Entrega 1**, que rindes al cierre de esta semana.

Ten abierta en paralelo la planilla `S2_Bitacora_Decisiones_CDD.xlsx`. Escríbela **mientras trabajas**, no al final: reconstruir después por qué tomaste una decisión es mucho más difícil de lo que parece.

In [15]:
import pandas as pd
import numpy as np

RUTA = ''
SEMILLA = 20262
np.random.seed(SEMILLA)

ARCHIVO = '20260303_Resumen_Rendimiento_2025_20260131.csv'   # <- reemplaza
SEPARADOR = ','                        # <- ajusta si corresponde

df = pd.read_csv(RUTA + ARCHIVO, sep=SEPARADOR)
print('Forma:', df.shape)
df.head()

Forma: (15897, 216)


,AGNO,RBD,DGV_RBD,NOM_RBD,COD_REG_RBD,NOM_REG_RBD_A,COD_PRO_RBD,COD_COM_RBD,NOM_COM_RBD,COD_DEPROV_RBD,...,PROM_ASIS_APR_SI,PROM_ASIS_APR_HOM,PROM_ASIS_APR_MUJ,PROM_ASIS_NB,PROM_ASIS_REP_SI,PROM_ASIS_REP_HOM,PROM_ASIS_REP_MUJ,PROM_ASIS_APR,PROM_ASIS_REP,PROM_ASIS
0,2025,1,9,LICEO POLITECNICO ARICA,15,AYP,151,15101,ARICA,151,...,NaN,90.500000,95.625000,NaN,NaN,NaN,NaN,94.600000,NaN,94.600000
1,2025,1,9,LICEO POLITECNICO ARICA,15,AYP,151,15101,ARICA,151,...,NaN,89.030769,87.576923,NaN,NaN,59.771429,53.3125,88.574661,57.745098,86.372549
2,2025,1,9,LICEO POLITECNICO ARICA,15,AYP,151,15101,ARICA,151,...,NaN,89.385542,87.344828,NaN,NaN,67.636364,70.0000,88.857143,67.833333,86.822581
3,2025,1,9,LICEO POLITECNICO ARICA,15,AYP,151,15101,ARICA,151,...,NaN,90.800000,86.477273,NaN,NaN,36.500000,41.5000,87.277778,39.000000,85.553571
4,2025,1,9,LICEO POLITECNICO ARICA,15,AYP,151,15101,ARICA,151,...,NaN,67.000000,86.395349,NaN,NaN,NaN,NaN,85.130435,NaN,85.130435


In [16]:
df.dtypes

,0
AGNO,int64
RBD,int64
DGV_RBD,int64
NOM_RBD,object
COD_REG_RBD,int64
...,...
PROM_ASIS_REP_HOM,float64
PROM_ASIS_REP_MUJ,float64
PROM_ASIS_APR,float64
PROM_ASIS_REP,float64


## Parte 1 · Diagnóstico de las cinco dimensiones

Antes de limpiar nada, hay que saber qué tienes. Completa la hoja **«Resumen de calidad»** de la bitácora con los resultados de esta parte.

### 1.1 Completitud

In [17]:
faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,pct
PROM_ASIS_REP_SI,15896,99.99
PROM_ASIS_NB,15889,99.95
PROM_ASIS_APR_SI,15882,99.91
PROM_ASIS_REP_MUJ,9437,59.36
PROM_ASIS_REP_HOM,7953,50.03
...,...,...
APR_NB,0,0.00
TRA_HOM_TO,0,0.00
SI_MUJ_TO,0,0.00
TRA_MUJ_TO,0,0.00


### ✏️ Actividad 1.1

Elige la columna con más faltantes y averigua **si se reparten al azar o se concentran en algún grupo**. Usa `pd.crosstab` contra alguna columna categórica, como en el Ejercicio 1 del cuaderno resuelto.

Es la diferencia entre poder imputar y tener que declarar.

In [ ]:
# 1. Seleccionamos la columna con más faltantes analizada en 1.1
columna_mas_faltantes = 'PROM_ASIS_REP_SI'

# 2. Elegimos una columna categórica de control, por ejemplo, la Región
columna_categorica = 'NOM_REG_RBD_A'

# 3. Creamos una columna temporal booleana (True si es nulo, False si no)
df['es_nulo'] = df[columna_mas_faltantes].isna()

# 4. Cruzamos las variables usando pd.crosstab en porcentajes por fila (normalize='index')
tabla_cruce = pd.crosstab(df[columna_categorica], df['es_nulo'], normalize='index') * 100

print(f"--- Porcentaje de faltantes en '{columna_mas_faltantes}' por '{columna_categorica}' ---")
display(tabla_cruce.round(2))

# 5. Limpiamos la columna temporal para dejar el DataFrame intacto
df.drop(columns=['es_nulo'], inplace=True)

--- Porcentaje de faltantes en 'PROM_ASIS_REP_SI' por 'NOM_REG_RBD_A' ---


es_nulo,False,True
NOM_REG_RBD_A,,
ANTOF,0.00,100.00
ARAUC,0.00,100.00
ATCMA,0.00,100.00
AYP,0.00,100.00
AYSEN,0.00,100.00
BBIO,0.00,100.00
COQ,0.00,100.00
LAGOS,0.00,100.00
LGBO,0.00,100.00


### 1.2 Exactitud — rangos y valores imposibles

In [ ]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
AGNO,15897.0,2025.00,0.00,2025.0,2025.00,2025.00,2025.00,2025.0
RBD,15897.0,12676.84,10079.00,1.0,4594.00,9992.00,18251.00,42395.0
DGV_RBD,15897.0,4.47,2.88,0.0,2.00,4.00,7.00,9.0
COD_REG_RBD,15897.0,9.24,3.88,1.0,6.00,9.00,13.00,16.0
COD_PRO_RBD,15897.0,94.44,38.51,11.0,61.00,91.00,131.00,163.0
...,...,...,...,...,...,...,...,...
PROM_ASIS_REP_HOM,7944.0,70.44,20.33,0.0,60.39,74.71,85.00,100.0
PROM_ASIS_REP_MUJ,6460.0,66.37,22.26,0.0,53.00,70.26,84.00,100.0
PROM_ASIS_APR,15756.0,89.46,12.13,0.0,87.89,91.17,94.65,100.0
PROM_ASIS_REP,9130.0,68.64,20.25,0.0,58.00,72.33,83.50,100.0


### ✏️ Actividad 1.2

Mira los mínimos y máximos. **¿Hay valores imposibles para lo que la columna representa?** Busca en particular valores centinela: `-1`, `0`, `999`, `9999`, `-999`, o cadenas como `'S/I'`, `'NULL'`, `'sin dato'`.

Recuerda: un centinela numérico **no** se delata en los tipos de dato.

In [ ]:
# Actividad 1.2 - Búsqueda automatizada de valores imposibles y centinelas
import pandas as pd
import numpy as np

# Validar de forma segura si 'df' existe antes de iniciar el análisis
if 'df' not in locals() and 'df' not in globals():
    print("❌ ERROR: La variable 'df' no está en memoria.")
    print("👉 Por favor, ve arriba en tu notebook y ejecuta la celda 'z9Kq9EHND33d' para cargar tus datos antes de correr esta celda.")
else:
    # 1. Definimos los sospechosos habituales de ser valores centinelas numéricos y de texto
    centinelas_numericos = [-1, -99, -999, 999, 9999, 99999]
    centinelas_texto = ['S/I', 'SI', 'S/D', 'NULL', 'NONE', 'NAN', 'SIN DATO', 'SIN REGISTRO', ' ', '']

    hallazgos = []

    # 2. Análisis de Columnas Numéricas
    for col in df.select_dtypes(include='number').columns:
        valores_unicos = df[col].dropna().unique()

        # Buscar si contiene números sospechosos centinelas
        coincidencias = [v for v in valores_unicos if v in centinelas_numericos]
        if coincidencias:
            hallazgos.append({
                'Columna': col,
                'Tipo': 'Numérico (Centinela Exacto)',
                'Detalle': f"Contiene valores sospechosos: {coincidencias}"
            })

        # Alerta lógica: Porcentajes de asistencia fuera de rango (0-100)
        if 'PROM_ASIS' in col or 'PORCENTAJE' in col:
            fuera_de_rango = [v for v in valores_unicos if v < 0.0 or v > 100.0]
            if fuera_de_rango:
                hallazgos.append({
                    'Columna': col,
                    'Tipo': 'Lógico (Asistencia fuera de 0-100)',
                    'Detalle': f"Valores fuera de rango: {fuera_de_rango[:5]}"
                })

    # 3. Análisis de Columnas de Texto
    for col in df.select_dtypes(include='object').columns:
        valores_limpios = df[col].astype(str).str.strip().str.upper().unique()
        coincidencias = [v for v in valores_limpios if v in centinelas_texto]
        if coincidencias:
            hallazgos.append({
                'Columna': col,
                'Tipo': 'Texto (Posible Nulo Oculto)',
                'Detalle': f"Términos encontrados: {coincidencias}"
            })

    # 4. Presentación de Resultados
    if hallazgos:
        df_reporte = pd.DataFrame(hallazgos)
        print("⚠️ SE DETECTARON POSIBLES VALORES IMPOSIBLES O CENTINELAS EN TUS DATOS:")
        display(df_reporte)
    else:
        print("✅ ¡Excelente! No se detectaron valores centinelas obvios en el dataset.")

    # 5. Reporte rápido de extremos para variables críticas
    print("\n--- Control de extremos para variables clave ---")
    variables_clave = [c for c in ['PROM_ASIS', 'RET_HOM_TO', 'RET_MUJ_TO', 'COD_DEPE'] if c in df.columns]
    display(df[variables_clave].describe().T[['min', 'max']])

⚠️ SE DETECTARON POSIBLES VALORES IMPOSIBLES O CENTINELAS EN TUS DATOS:


,Columna,Tipo,Detalle
0,RBD,Numérico (Centinela Exacto),"Contiene valores sospechosos: [np.int64(999), ..."



--- Control de extremos para variables clave ---


,min,max
PROM_ASIS,0.0,100.0
RET_HOM_TO,0.0,344.0
RET_MUJ_TO,0.0,197.0
COD_DEPE,1.0,6.0


In [ ]:
# Código removido: sección de bitácora Excel omitida por solicitud del usuario.

In [ ]:
# Código de actualización de bitácora removido.

In [ ]:
# Código de descarga de bitácora removido.

In [ ]:
# Reestructuración de bitácora omitida por solicitud.

In [ ]:
# Descarga automatizada omitida.

In [ ]:
# Reversión de bitácora omitida.

### 1.3 Consistencia — reglas cruzadas

### ✏️ Actividad 1.3

Escribe **dos reglas de consistencia** que tu conjunto debería cumplir y verifícalas. Ejemplos:

- «La fecha de término no puede ser anterior a la de inicio.»
- «Si el estado es *cerrado*, el monto vigente debe estar vacío.»
- «La suma de las partes debe igualar el total.»

Estas reglas no salen de los datos: salen de entender qué representan.

In [ ]:
# Regla 1: El promedio de asistencia general debe estar estrictamente entre 0 y 100
inconsistencias_1 = df[(df['PROM_ASIS'] < 0) | (df['PROM_ASIS'] > 100)]
print(f"Regla 1 - Asistencias fuera de rango (0-100): {len(inconsistencias_1)} casos")

# Regla 2: El promedio general de aprobados (PROM_ASIS_APR) debe estar acotado entre el promedio de hombres y mujeres
# (Solo evaluamos filas donde existan los tres valores)
filtro_generos = df['PROM_ASIS_APR_HOM'].notna() & df['PROM_ASIS_APR_MUJ'].notna() & df['PROM_ASIS_APR'].notna()
df_eval = df[filtro_generos]

# El promedio total no puede ser estrictamente mayor que ambos ni estrictamente menor que ambos
inconsistencias_2 = df_eval[
    (df_eval['PROM_ASIS_APR'] > df_eval[['PROM_ASIS_APR_HOM', 'PROM_ASIS_APR_MUJ']].max(axis=1)) |
    (df_eval['PROM_ASIS_APR'] < df_eval[['PROM_ASIS_APR_HOM', 'PROM_ASIS_APR_MUJ']].min(axis=1))
]
print(f"Regla 2 - Inconsistencias de promedios cruzados por género: {len(inconsistencias_2)} casos")




Regla 1 - Asistencias fuera de rango (0-100): 0 casos
Regla 2 - Inconsistencias de promedios cruzados por género: 1 casos


### 1.4 Unicidad

In [ ]:
print('Duplicados exactos:', df.duplicated().sum())
# Si tu conjunto tiene una columna identificadora, revísala también:
# print('ID repetidos:', df['TU_ID'].duplicated().sum())

Duplicados exactos: 0


### ✏️ Actividad 1.4

Construye una **clave normalizada** (sin espacios, en mayúsculas, sin tildes) con las columnas que deberían identificar un registro, y cuenta los casi-duplicados.

In [ ]:
# Actividad 1.4 - Construcción y validación de una clave única normalizada
import pandas as pd
import numpy as np

def normalizar_columna(serie):
    """Limpia, elimina espacios y asegura formato string sin decimales flotantes."""
    return serie.fillna('').astype(str).str.strip().str.replace(r'\.0$', '', regex=True).str.upper()

if 'df' in globals():
    # 1. Normalizamos rigurosamente cada componente de nuestra clave
    key_rbd = normalizar_columna(df['RBD'])
    key_agno = normalizar_columna(df['AGNO'])
    key_ense = normalizar_columna(df['COD_ENSE'])

    # 2. Construimos la clave única compuesta limpia
    df['CLAVE_UNICA_NORMALIZADA'] = key_rbd + "_" + key_agno + "_" + key_ense

    # 3. Calculamos métricas de unicidad
    duplicados_clave_compuesta = df['CLAVE_UNICA_NORMALIZADA'].duplicated().sum()
    total_filas = len(df)

    print(f"--- RESULTADOS DEL ANÁLISIS DE UNICIDAD (NORMALIZADO) ---")
    print(f"Total de registros en el dataset: {total_filas}")
    print(f"Registros con clave duplicada (RBD + AGNO + COD_ENSE): {duplicados_clave_compuesta}")

    # 4. Mostrar el único caso duplicado si existe para diagnosticarlo
    if duplicados_clave_compuesta > 0:
        print("\nDetalle del registro duplicado encontrado:")
        filas_duplicadas = df[df['CLAVE_UNICA_NORMALIZADA'].duplicated(keep=False)]
        display(filas_duplicadas[['CLAVE_UNICA_NORMALIZADA', 'RBD', 'AGNO', 'COD_ENSE', 'NOM_RBD', 'PROM_ASIS']].head(10))
else:
    print("La variable 'df' no está disponible en memoria.")

--- RESULTADOS DEL ANÁLISIS DE UNICIDAD (NORMALIZADO) ---
Total de registros en el dataset: 15767
Registros con clave duplicada (RBD + AGNO + COD_ENSE): 1

Detalle del registro duplicado encontrado:


,CLAVE_UNICA_NORMALIZADA,RBD,AGNO,COD_ENSE,NOM_RBD,PROM_ASIS
924,_2025_110,NaN,2025,110,ESCUELA PARTICULAR 18 DE SEPTIEMBRE,92.000000
7954,_2025_110,NaN,2025,110,ESCUELA ANTUMALAL,86.775701


In [ ]:
def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

# 1. Definimos las columnas que en teoría deberían identificar de forma única a cada registro
# Para un resumen anual escolar, la combinación de RBD (Establecimiento) y AGNO (Año) debería ser única.

# 2. Creamos una clave normalizada concatenando ambas columnas limpias
clave_unica = normalizar_texto(df['RBD']) + "_" + normalizar_texto(df['AGNO'])

# 3. Contamos cuántos casi-duplicados (mismo colegio el mismo año) existen
duplicados_clave = clave_unica.duplicated().sum()

print(f"Cantidad de registros con clave duplicada (RBD + Año): {duplicados_clave}")
if duplicados_clave > 0:
    print("\nEjemplo de registros que comparten la misma clave:")
    display(df[clave_unica.isin(clave_unica[clave_unica.duplicated()].head(3))].sort_values(by='RBD')[['RBD', 'AGNO', 'NOM_RBD']].head(6))

Cantidad de registros con clave duplicada (RBD + Año): 5244

Ejemplo de registros que comparten la misma clave:


,RBD,AGNO,NOM_RBD
0,1.0,2025,LICEO POLITECNICO ARICA
1,1.0,2025,LICEO POLITECNICO ARICA
2,1.0,2025,LICEO POLITECNICO ARICA
3,1.0,2025,LICEO POLITECNICO ARICA
4,1.0,2025,LICEO POLITECNICO ARICA


### 1.5 Oportunidad

In [ ]:
# 1. Analizamos los años presentes en el conjunto de datos
anios_presentes = sorted(df['AGNO'].unique())
print(f"Años académicos registrados en el conjunto de datos: {anios_presentes}")

# 2. Contamos cuántos registros hay por cada año para evaluar la distribución temporal
distribucion_temporal = df['AGNO'].value_counts().sort_index()
print("\nDistribución de registros por año:")
display(distribucion_temporal.to_frame('cantidad_registros'))

# 3. Verificamos si existe alguna otra columna con indicación temporal o fecha de carga
columnas_fecha = [col for col in df.columns if 'FECHA' in col.upper() or 'DATE' in col.upper()]
if columnas_fecha:
    print(f"\nColumnas tipo fecha identificadas: {columnas_fecha}")
    for col in columnas_fecha:
        print(f"Rango de {col}: {df[col].min()} hasta {df[col].max()}")
else:
    print("\nNo se identificaron columnas explícitas de tipo fecha (timestamp/date) adicionales a 'AGNO'.")

Años académicos registrados en el conjunto de datos: [np.int64(2025)]

Distribución de registros por año:


,cantidad_registros
AGNO,
2025,15767



No se identificaron columnas explícitas de tipo fecha (timestamp/date) adicionales a 'AGNO'.


---
## Parte 2 · Limpieza, decisión por decisión

Ahora corriges. **Regla del laboratorio: cada celda de esta parte tiene su fila en la bitácora.** Si aplicas una transformación que no registraste, no cuenta.

Recuerda las tres decisiones legítimas: **Corregir**, **Descartar** o **Declarar como limitación**.

### ✏️ Actividad 2.1 — Centinelas a faltantes

In [ ]:
# Actividad 2.1 - Tratamiento de nulos y centinelas para nuestra pregunta de investigación

# 1. Conservamos los registros válidos de PROM_ASIS ya que representa nuestra variable central de Rendimiento
# Dado que los nulos en PROM_ASIS representan solo el 0.82% (130 registros), la eliminación directa es la opción más segura.
print(f"Filas antes de limpiar PROM_ASIS: {len(df)}")
df = df.dropna(subset=['PROM_ASIS'])
print(f"Filas después de limpiar PROM_ASIS: {len(df)}")

# 2. Documentamos formalmente que las columnas con altos porcentajes de nulos no se imputarán
# PROM_ASIS_REP_HOM (50.03% nulos) y PROM_ASIS_REP_SI (99.99% nulos) se mantienen como NaN por ser vacíos estructurales lógicos
columnas_limite = ['PROM_ASIS_REP_HOM', 'PROM_ASIS_REP_SI']
print(f"\nTratamiento registrado en bitácora para {columnas_limite}: DECLARAR COMO LIMITACIÓN ESTRUCTURAL (Sin imputar)")

Filas antes de limpiar PROM_ASIS: 15767
Filas después de limpiar PROM_ASIS: 15767

Tratamiento registrado en bitácora para ['PROM_ASIS_REP_HOM', 'PROM_ASIS_REP_SI']: DECLARAR COMO LIMITACIÓN ESTRUCTURAL (Sin imputar)


### ✏️ Actividad 2.2 — Normalización de categorías

Normaliza mecánicamente y después revisa lo que queda: los plurales, las abreviaturas y los sinónimos necesitan un diccionario de equivalencias que escribes tú.

In [ ]:
# Actividad 2.2 - Normalización de la variable de dependencia administrativa

# 1. Inspeccionamos los valores únicos de 'COD_DEPE' que representa la dependencia del establecimiento
valores_depe = df['COD_DEPE'].unique()
print(f"Códigos de dependencia administrativa únicos en el dataset: {valores_depe}")

# 2. Mostramos el conteo de escuelas por cada tipo de dependencia administrativa para verificar la distribución
distribucion_depe = df['COD_DEPE'].value_counts().sort_index()
print("\nDistribución por dependencia administrativa (COD_DEPE):")
display(distribucion_depe.to_frame('cantidad_establecimientos'))

Códigos de dependencia administrativa únicos en el dataset: [6 3 4 5 2 1]

Distribución por dependencia administrativa (COD_DEPE):


,cantidad_establecimientos
COD_DEPE,
1,665
2,3746
3,7742
4,973
5,159
6,2482


### ✏️ Actividad 2.3 — Duplicados

In [ ]:
# Actividad 2.3 - Tratamiento de registros duplicados por RBD y Año

# 1. Comprobamos la presencia de la columna 'COD_ENSE' o 'COD_ENSE2' (código de enseñanza)
# para verificar si los duplicados corresponden a diferentes niveles educativos dentro de un mismo colegio
columnas_nivel = [col for col in df.columns if 'COD_ENSE' in col]
print(f"Columnas de nivel educativo encontradas: {columnas_nivel}")

# 2. Mostramos un ejemplo real de un colegio (RBD) que tiene múltiples filas el mismo año
# para justificar nuestra decisión de mantenerlas separadas o consolidarlas (corregido 'False')
rbd_ejemplo = df[df.duplicated(subset=['RBD', 'AGNO'], keep=False)]['RBD'].iloc[0] if len(df[df.duplicated(subset=['RBD', 'AGNO'])]) > 0 else None

if rbd_ejemplo:
    print(f"\nRegistros para el establecimiento RBD {rbd_ejemplo} en el año 2025:")
    columnas_mostrar = ['RBD', 'AGNO', 'NOM_RBD'] + [col for col in ['COD_ENSE', 'COD_ENSE2', 'PROM_ASIS'] if col in df.columns]
    display(df[df['RBD'] == rbd_ejemplo][columnas_mostrar])
else:
    print("No se encontraron duplicados clave bajo los filtros aplicados.")

Columnas de nivel educativo encontradas: ['COD_ENSE', 'COD_ENSE2']

Registros para el establecimiento RBD 1 en el año 2025:


,RBD,AGNO,NOM_RBD,COD_ENSE,COD_ENSE2,PROM_ASIS
0,1,2025,LICEO POLITECNICO ARICA,463,8,94.600000
1,1,2025,LICEO POLITECNICO ARICA,510,7,86.372549
2,1,2025,LICEO POLITECNICO ARICA,563,8,86.822581
3,1,2025,LICEO POLITECNICO ARICA,610,7,85.553571
4,1,2025,LICEO POLITECNICO ARICA,663,8,85.130435


### ✏️ Actividad 2.4 — Tipos de dato

Convierte a numérico lo que deba ser numérico y a fecha lo que deba ser fecha. Si hay formatos mixtos, prueba formatos explícitos en orden en vez de dejar que pandas adivine.

In [ ]:
# Actividad 2.4 - Validación y corrección de tipos de datos de las variables del estudio

# 1. Definimos las variables clave de nuestra pregunta de investigación
variables_clave = ['PROM_ASIS', 'RET_HOM_TO', 'RET_MUJ_TO', 'COD_DEPE']

print("--- Tipos de datos iniciales de las variables clave ---")
print(df[variables_clave].dtypes)

# 2. Nos aseguramos de que las tasas de retiro y asistencia sean de tipo numérico flotante,
# y el código de dependencia sea un entero o categoría según corresponda
for col in ['PROM_ASIS', 'RET_HOM_TO', 'RET_MUJ_TO']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['COD_DEPE'] = df['COD_DEPE'].astype('int64')

print("\n--- Tipos de datos confirmados después del casteo ---")
print(df[variables_clave].dtypes)

--- Tipos de datos iniciales de las variables clave ---
PROM_ASIS     float64
RET_HOM_TO      int64
RET_MUJ_TO      int64
COD_DEPE        int64
dtype: object

--- Tipos de datos confirmados después del casteo ---
PROM_ASIS     float64
RET_HOM_TO      int64
RET_MUJ_TO      int64
COD_DEPE        int64
dtype: object


### ✏️ Actividad 2.5 — Atípicos

Identifica los valores extremos y **pronúnciate sobre cada grupo**: ¿error de registro o fenómeno real?

Cruza la variable con otra que te permita juzgar plausibilidad, como en el Ejercicio 8. No elimines atípicos solo por serlo.

In [ ]:
# Actividad 2.5 - Análisis de valores atípicos (outliers)

# 1. Calculamos las estadísticas clave de retiro para hombres y mujeres
print("--- Distribución de Retiros Totales por Establecimiento ---")
display(df[['RET_HOM_TO', 'RET_MUJ_TO', 'PROM_ASIS']].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]).round(2))

# 2. Identificamos casos extremos en tasas de retiro (ej. sobre el percentil 99)
umbral_retiro_hom = df['RET_HOM_TO'].quantile(0.99)
umbral_retiro_muj = df['RET_MUJ_TO'].quantile(0.99)

extremos_retiro = df[(df['RET_HOM_TO'] > umbral_retiro_hom) | (df['RET_MUJ_TO'] > umbral_retiro_muj)]

print(f"\nSe identificaron {len(extremos_retiro)} registros con retiros extremadamente altos (P99 Hom: {umbral_retiro_hom}, P99 Muj: {umbral_retiro_muj})")
print("Ejemplo de establecimientos con retiros extremos y su asistencia promedio correspondientes:")
display(extremos_retiro[['NOM_RBD', 'COD_DEPE', 'RET_HOM_TO', 'RET_MUJ_TO', 'PROM_ASIS']].head(10))

--- Distribución de Retiros Totales por Establecimiento ---


,RET_HOM_TO,RET_MUJ_TO,PROM_ASIS
count,15767.00,15767.00,15767.00
mean,2.81,1.99,88.77
std,7.63,4.94,12.40
min,0.00,0.00,0.00
1%,0.00,0.00,13.70
25%,0.00,0.00,87.20
50%,1.00,1.00,90.67
75%,3.00,2.00,94.22
95%,11.00,8.00,99.27
99%,30.00,21.00,100.00



Se identificaron 222 registros con retiros extremadamente altos (P99 Hom: 30.0, P99 Muj: 21.0)
Ejemplo de establecimientos con retiros extremos y su asistencia promedio correspondientes:


,NOM_RBD,COD_DEPE,RET_HOM_TO,RET_MUJ_TO,PROM_ASIS
2,LICEO POLITECNICO ARICA,6,66,15,86.822581
5,ESC. PEDRO VICENTE GUTIERREZ TORRES,6,149,1,97.168865
6,ESC. PEDRO VICENTE GUTIERREZ TORRES,6,38,0,98.879518
94,INST. COM. DE IQUIQUE BALDOMERO WOLNITZKY,6,21,22,86.881092
220,CENTRO EDUC.INTEGRAL DE ADULTOS C.E.I.A,6,83,56,57.327273
225,ESCUELA BASICA PRESIDENTE BALMACEDA,6,99,69,91.002091
227,ESCUELA REPUBLICA DE CHILE,6,22,26,92.510112
262,LICEO INDUSTRIAL EULOGIO GORDO MONEO,1,94,32,66.310651
263,LICEO COMERCIAL JERARDO MUNOZ CAMPOS,1,40,25,86.934769
268,LICEO TECNICO ANTOFAGASTA,1,21,81,80.054688


In [7]:
import pandas as pd
import numpy as np
import os

# Recuperar el dataframe de manera robusta usando variables globales o cargando el archivo original directamente
archivo_original = '20260303_Resumen_Rendimiento_2025_20260131.csv'

if 'limpio' in globals():
    data_analisis = limpio
elif 'df' in globals():
    data_analisis = df
elif os.path.exists('mi_conjunto_limpio.csv'):
    data_analisis = pd.read_csv('mi_conjunto_limpio.csv', sep=';')
elif os.path.exists(archivo_original):
    # Carga de respaldo y limpieza rápida inline de la variable objetivo
    data_analisis = pd.read_csv(archivo_original, sep=',')
    data_analisis = data_analisis.dropna(subset=['PROM_ASIS'])
else:
    raise NameError("No se encontró ningún archivo de datos. Por favor sube tu archivo CSV al entorno de Colab.")

# 1. Cálculo de estadísticas descriptivas completas y cuartiles (Q1, Q2/Mediana, Q3)
variables_analisis = ['PROM_ASIS', 'RET_HOM_TO', 'RET_MUJ_TO']
estadisticas = data_analisis[variables_analisis].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T

# Calcular el Rango Intercuartílico (IQR) y los límites para detectar atípicos
estadisticas['IQR'] = estadisticas['75%'] - estadisticas['25%']
estadisticas['Limite_Superior_IQR'] = estadisticas['75%'] + 1.5 * estadisticas['IQR']
estadisticas['Limite_Inferior_IQR'] = np.clip(estadisticas['25%'] - 1.5 * estadisticas['IQR'], 0, None)

print("=== ESTADÍSTICA DESCRIPTIVA COMPLETA Y LÍMITES IQR ===")
display(estadisticas.round(2))

=== ESTADÍSTICA DESCRIPTIVA COMPLETA Y LÍMITES IQR ===


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max,IQR,Limite_Superior_IQR,Limite_Inferior_IQR
PROM_ASIS,15767.0,88.77,12.40,0.0,87.2,90.67,94.22,97.67,99.27,100.0,100.0,7.01,104.74,76.68
RET_HOM_TO,15767.0,2.81,7.63,0.0,0.0,1.00,3.00,7.00,11.00,30.0,344.0,3.00,7.50,0.00
RET_MUJ_TO,15767.0,1.99,4.94,0.0,0.0,1.00,2.00,5.00,8.00,21.0,197.0,2.00,5.00,0.00


In [21]:
import pandas as pd
import numpy as np
import os

# Recuperar el dataframe de manera robusta
archivo_original = '20260303_Resumen_Rendimiento_2025_20260131.csv'

if 'limpio' in globals():
    data_analisis = limpio
elif 'df' in globals():
    data_analisis = df
elif os.path.exists('mi_conjunto_limpio.csv'):
    data_analisis = pd.read_csv('mi_conjunto_limpio.csv', sep=';')
elif os.path.exists(archivo_original):
    data_analisis = pd.read_csv(archivo_original, sep=',')
    data_analisis = data_analisis.dropna(subset=['PROM_ASIS'])
else:
    raise NameError("No se encontró ningún archivo de datos para el análisis de plausibilidad.")

# 1. Definimos los límites del Percentil 99
umbral_hom = data_analisis['RET_HOM_TO'].quantile(0.99)
umbral_muj = data_analisis['RET_MUJ_TO'].quantile(0.99)

# 2. Creamos la variable booleana de atípicos
data_analisis['Es_Atipico_Retiro'] = (data_analisis['RET_HOM_TO'] > umbral_hom) | (data_analisis['RET_MUJ_TO'] > umbral_muj)

# 3. Mapeamos True/False a etiquetas descriptivas para mejor lectura
data_analisis['Estado_Retiro'] = data_analisis['Es_Atipico_Retiro'].map({False: '1. Normal', True: '2. Atípico (Altos Retiros)'})

# 4. Agrupamos por Nivel Educativo (COD_ENSE) usando las etiquetas nuevas
print("=== DISTRIBUCIÓN DE CASOS CON RETIROS EXTREMOS POR TIPO DE ENSEÑANZA ===")
cruce_ense = pd.crosstab(data_analisis['COD_ENSE'], data_analisis['Estado_Retiro'], normalize='columns') * 100
display(cruce_ense.round(2))

# 5. Calculamos el promedio de asistencia con las etiquetas descriptivas
print("\n=== PROMEDIO DE ASISTENCIA SEGÚN COMPORTAMIENTO DE RETIROS ===")
prom_asistencia = data_analisis.groupby('Estado_Retiro')['PROM_ASIS'].mean().to_frame('Asistencia_Promedio')
display(prom_asistencia.round(2))

# Limpieza de columnas temporales
data_analisis.drop(columns=['Es_Atipico_Retiro', 'Estado_Retiro'], inplace=True)

=== DISTRIBUCIÓN DE CASOS CON RETIROS EXTREMOS POR TIPO DE ENSEÑANZA ===


Estado_Retiro,1. Normal,2. Atípico (Altos Retiros)
COD_ENSE,,
110,50.09,4.50
165,2.06,11.71
167,0.62,2.70
211,0.06,0.00
212,3.02,0.90
213,0.05,0.00
214,9.42,1.80
215,0.06,0.00
216,0.35,0.00



=== PROMEDIO DE ASISTENCIA SEGÚN COMPORTAMIENTO DE RETIROS ===


,Asistencia_Promedio
Estado_Retiro,
1. Normal,88.82
2. Atípico (Altos Retiros),85.28


### Conclusión sobre los Atípicos (Actividad 2.5)

*1. Análisis de Límites y Rangos (IQR vs. Percentiles)
El criterio estándar de Rango Intercuartílico ($IQR$$IQR$) propone límites superiores muy bajos: 7.50 retiros para hombres y 5.00 para mujeres. Calificar como "error" a cualquier colegio por encima de estos números sería un error metodológico grave.
Al aplicar el criterio del Percentil 99 (establecimientos con más de 30 retiros masculinos o 21 retiros femeninos), aislamos los casos verdaderamente extremos.
2. Pronunciamiento sobre los Atípicos: ¿Error de registro o Fenómeno Real?
Se concluye que estos registros representan un FENÓMENO REAL debido a dos criterios de plausibilidad cruzada:

Plausibilidad de Modalidad Educativa (COD_ENSE): Al cruzar las variables, observamos que el 61.26% de todos los casos con retiros atípicos se concentra en la modalidad 363 (Educación Media Técnico-Profesional de Adultos). En esta modalidad, debido a motivos laborales, familiares o de vulnerabilidad social, la movilidad y deserción escolar es un fenómeno estructural conocido. No se trata de un error de digitación, sino de la naturaleza de los alumnos que asisten a esos centros.
Plausibilidad por Consistencia de Asistencia (PROM_ASIS): Los colegios sin retiros extremos promedian 88.82% de asistencia, mientras que los establecimientos con retiros extremos experimentan una caída lógica en su promedio de asistencia, situándose en 85.28%. Esta consistencia interna valida la veracidad del registro.
3. Decisión para la Bitácora
Acción: CONSERVAR los datos atípicos.
Justificación: Eliminar estos registros eliminaría la información más valiosa sobre los contextos críticos de deserción escolar, sesgando cualquier análisis o modelo predictivo de rendimiento que desarrollemos en las próximas semanas.


---
## Parte 3 · Transformación y nuevos atributos

Preparar no es solo corregir: también es **construir** lo que el análisis va a necesitar.

### ✏️ Actividad 3.1 — Un atributo derivado

Crea **al menos una variable nueva** a partir de las existentes, que aporte a tu pregunta analítica. Ejemplos: una razón entre dos columnas, la antigüedad en años a partir de una fecha, un tramo a partir de un valor continuo.

Explica en la celda siguiente por qué esa variable ayuda a responder tu pregunta.

In [10]:
# Actividad 3.1 - Creación de Atributos Derivados para responder la Pregunta de Investigación
import pandas as pd
import os

# Recuperar el dataframe limpio de manera robusta
if 'limpio' in globals():
    data_analisis = limpio
elif 'df' in globals():
    data_analisis = df
elif os.path.exists('mi_conjunto_limpio.csv'):
    data_analisis = pd.read_csv('mi_conjunto_limpio.csv', sep=';')
else:
    archivo_original = '20260303_Resumen_Rendimiento_2025_20260131.csv'
    data_analisis = pd.read_csv(archivo_original, sep=',')
    data_analisis = data_analisis.dropna(subset=['PROM_ASIS'])

# 1. Atributo 1: Retiro Total Combinado (Suma de retiros masculinos y femeninos)
data_analisis['RET_TOTAL_TO'] = data_analisis['RET_HOM_TO'] + data_analisis['RET_MUJ_TO']

# 2. Atributo 2: Tramo de Asistencia (Categorización según políticas de alerta ministerial en Chile)
def categorizar_asistencia(asis):
    if asis < 85.0:
        return '1. Grave (<85%)'
    elif asis < 90.0:
        return '2. Baja (85-90%)'
    elif asis < 95.0:
        return '3. Regular (90-95%)'
    else:
        return '4. Ideal (95-100%)'

data_analisis['TRAMO_ASISTENCIA'] = data_analisis['PROM_ASIS'].apply(categorizar_asistencia)

# Sincronizar de vuelta al namespace global
limpio = data_analisis

# 3. Validamos las nuevas variables creadas
print("--- Estadísticas descriptivas del Retiro Total Combinado (RET_TOTAL_TO) ---")
print(limpio['RET_TOTAL_TO'].describe().round(2))

print("\n--- Distribución de escuelas por Tramo de Asistencia ---")
display(limpio['TRAMO_ASISTENCIA'].value_counts().sort_index().to_frame('cantidad_registros'))

--- Estadísticas descriptivas del Retiro Total Combinado (RET_TOTAL_TO) ---
count    15767.00
mean         4.81
std         11.65
min          0.00
25%          0.00
50%          2.00
75%          5.00
max        541.00
Name: RET_TOTAL_TO, dtype: float64

--- Distribución de escuelas por Tramo de Asistencia ---


,cantidad_registros
TRAMO_ASISTENCIA,
1. Grave (<85%),2465
2. Baja (85-90%),4549
3. Regular (90-95%),5436
4. Ideal (95-100%),3317


### **Justificación Metodológica de los Atributos Derivados (Actividad 3.1)**

Ambas variables creadas aportan un valor analítico fundamental que optimiza el camino para responder nuestra pregunta de investigación sobre la relación entre el rendimiento/asistencia y la deserción escolar:

#### **1. Retiro Total Combinado (`RET_TOTAL_TO`)**
* **Sinergia y Reducción de Dimensionalidad:** Analizar los retiros de hombres (`RET_HOM_TO`) y mujeres (`RET_MUJ_TO`) de forma aislada duplica la complejidad de los futuros modelos predictivos y puede atomizar el impacto real del abandono. Al consolidar ambos en una variable única, capturamos la **magnitud absoluta de la deserción** por establecimiento y nivel, independientemente del género.
* **Robustez Estadística:** Minimiza el impacto de celdas con valor cero que se deben únicamente a la composición de matrícula por género (por ejemplo, liceos monogenéricos), ofreciendo un indicador de movilidad escolar mucho más estable y directamente correlacionable con la asistencia general.

#### **2. Tramos de Asistencia (`TRAMO_ASISTENCIA`)**
* **Relevancia Basada en Políticas Públicas:** En la legislación educativa (particularmente en Chile), el comportamiento de la asistencia no es un fenómeno lineal continuo. Existe un umbral crítico legal e institucional en el **85% de asistencia** (límite que define la promoción automática de curso y el cálculo de subvenciones estatales).
* **Interpretabilidad y Enfoque de Clasificación:** Al discretizar la variable continua `PROM_ASIS` en 4 categorías jerárquicas (*Grave, Baja, Regular e Ideal*), permitimos que el análisis posterior y los algoritmos predictivos de la Semana 4 se enfoquen en clasificar con alta precisión qué escuelas o niveles entrarán en zona de **Inasistencia Grave (<85%)**, facilitando la toma de decisiones preventivas en políticas públicas.

### ✏️ Actividad 3.2 — Normalización o codificación

Si tu conjunto tiene variables numéricas en escalas muy distintas, o categóricas que necesitarás codificar en la semana 4, déjalas preparadas aquí.

In [19]:
# Actividad 3.2 - Normalización y Codificación para Modelamiento (Semana 4)
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

if 'df' in globals():
    print("=============================================================")
    print("   PROCESAMIENTO Y ANÁLISIS DE ATRIBUTOS (ACTIVIDAD 3.2)   ")
    print("=============================================================")

    # 1. Copia de trabajo para evitar alteraciones conflictivas
    df_prep = df.copy()

    # Map oficial de dependencias en Chile para que el análisis sea legible
    map_depe = {
        '1': 'DEPE_Municipal',
        '2': 'DEPE_Particular_Subvencionado',
        '3': 'DEPE_Particular_Pagado',
        '4': 'DEPE_Corporacion_Administracion_Delegada',
        '5': 'DEPE_Servicio_Local_Educacion_SLEP',
        '6': 'DEPE_Particular_Subvencionado_Diferente'
    }

    # 2. Codificación One-Hot para Dependencia Administrativa
    if 'COD_DEPE' in df_prep.columns:
        print("\n[1/2] Codificando la variable categórica 'COD_DEPE' (One-Hot)...")
        df_prep['COD_DEPE'] = df_prep['COD_DEPE'].astype(str)
        # Generamos dummies
        df_prep = pd.get_dummies(df_prep, columns=['COD_DEPE'], prefix='DEPE', drop_first=True)

        # Renombramos las columnas dummy para que sean legibles en el análisis
        columnas_dummies = [c for c in df_prep.columns if c.startswith('DEPE_')]
        for col_dum in columnas_dummies:
            codigo = col_dum.split('_')[1]
            if codigo in map_depe:
                df_prep.rename(columns={col_dum: map_depe[codigo]}, inplace=True)
    else:
        print("\n⚠️ [1/2] 'COD_DEPE' ya fue codificada previamente.")
        # Corrección retrospectiva: Si ya existen las columnas DEPE_2, DEPE_3, etc., las renombramos directamente
        columnas_existentes = [c for c in df_prep.columns if c.startswith('DEPE_') and c.split('_')[1].isdigit()]
        if columnas_existentes:
            print("-> Renombrando columnas codificadas previas a nombres descriptivos para el reporte...")
            for col_dum in columnas_existentes:
                codigo = col_dum.split('_')[1]
                if codigo in map_depe:
                    df_prep.rename(columns={col_dum: map_depe[codigo]}, inplace=True)

    # 3. Escalamiento de variables de retiro mediante Min-Max Scaling (Rango 0 a 1)
    columnas_a_escalar = ['RET_HOM_TO', 'RET_MUJ_TO']
    print("\n[2/2] Escalando variables continuas de retiro al rango [0, 1] usando MinMaxScaler...")
    scaler = MinMaxScaler()

    for col in columnas_a_escalar:
        if col in df_prep.columns:
            df_prep[f'{col}_SCALED'] = scaler.fit_transform(df_prep[[col]])

    # 4. Reporte explicativo del resultado
    print("\n" + "="*60)
    print("          REPORTE DE INTERPRETACIÓN PARA TU ANÁLISIS")
    print("="*60)

    # Visualizar las columnas Dummy creadas
    cols_depe_nuevas = [col for col in df_prep.columns if col.startswith('DEPE_')]
    print(f"\n🟢 DEPENDENCIAS ADMINISTRATIVAS CODIFICADAS ({len(cols_depe_nuevas)} columnas creadas):")
    print("Nota: Se omitió una columna automáticamente (dummy trap) para servir como grupo de referencia en modelos.")
    display(df_prep[cols_depe_nuevas].head(5))

    # Visualizar estadísticas de escala
    cols_scaled_nuevas = [f'{col}_SCALED' for col in columnas_a_escalar if f'{col}_SCALED' in df_prep.columns]
    print("\n🟢 DISTRIBUCIÓN DE VARIABLES ESCALADAS (RETIROS POR GÉNERO):")
    print("Nota: 0.0 representa el colegio con menor cantidad de retiros; 1.0 el colegio con mayor deserción.")
    display(df_prep[cols_scaled_nuevas].describe().T[['min', 'max', 'mean', 'std']].round(4))

    # Guardar los cambios finales
    df = df_prep
    print("\n✅ ¡Actividad 3.2 completada de forma segura y lista para exportar!")
else:
    print("❌ Error: El DataFrame 'df' no está cargado en memoria. Ejecuta las celdas de la Parte 1 y 2.")

   PROCESAMIENTO Y ANÁLISIS DE ATRIBUTOS (ACTIVIDAD 3.2)   

⚠️ [1/2] 'COD_DEPE' ya fue codificada previamente.

[2/2] Escalando variables continuas de retiro al rango [0, 1] usando MinMaxScaler...

          REPORTE DE INTERPRETACIÓN PARA TU ANÁLISIS

🟢 DEPENDENCIAS ADMINISTRATIVAS CODIFICADAS (5 columnas creadas):
Nota: Se omitió una columna automáticamente (dummy trap) para servir como grupo de referencia en modelos.


,DEPE_Particular_Subvencionado,DEPE_Particular_Pagado,DEPE_Corporacion_Administracion_Delegada,DEPE_Servicio_Local_Educacion_SLEP,DEPE_Particular_Subvencionado_Diferente
0,False,False,False,False,True
1,False,False,False,False,True
2,False,False,False,False,True
3,False,False,False,False,True
4,False,False,False,False,True



🟢 DISTRIBUCIÓN DE VARIABLES ESCALADAS (RETIROS POR GÉNERO):
Nota: 0.0 representa el colegio con menor cantidad de retiros; 1.0 el colegio con mayor deserción.


,min,max,mean,std
RET_HOM_TO_SCALED,0.0,1.0,0.0082,0.0221
RET_MUJ_TO_SCALED,0.0,1.0,0.0102,0.0250



✅ ¡Actividad 3.2 completada de forma segura y lista para exportar!
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


---
## Parte 4 · El flujo como función reutilizable

Junta todo lo anterior en **una sola función**. Esto es lo que evalúa el RA3 y lo que se revisa en la Sección A de la Entrega Final: que tu limpieza se pueda volver a ejecutar de principio a fin.

In [22]:
def limpiar(ruta, sep=SEPARADOR, semilla=SEMILLA):
    """Flujo completo y reproducible de limpieza de mi conjunto de datos.

    Devuelve (df_limpio, informe).
    """
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import MinMaxScaler

    # Fijamos la semilla generadora
    np.random.seed(semilla)

    # Carga inicial de datos
    d = pd.read_csv(ruta, sep=sep)
    informe = {'filas_iniciales': len(d)}

    # 1. Tratamiento de nulos en variables clave (Actividad 2.1)
    # Eliminamos registros nulos en la variable objetivo principal de asistencia
    d = d.dropna(subset=['PROM_ASIS'])

    # 2. Normalización de categorías y codificación One-Hot (Actividad 2.2 y 3.2)
    map_depe = {
        '1': 'DEPE_Municipal',
        '2': 'DEPE_Particular_Subvencionado',
        '3': 'DEPE_Particular_Pagado',
        '4': 'DEPE_Corporacion_Administracion_Delegada',
        '5': 'DEPE_Servicio_Local_Educacion_SLEP',
        '6': 'DEPE_Particular_Subvencionado_Diferente'
    }

    if 'COD_DEPE' in d.columns:
        d['COD_DEPE'] = d['COD_DEPE'].astype(str)
        d = pd.get_dummies(d, columns=['COD_DEPE'], prefix='DEPE', drop_first=True)

        # Renombrar columnas creadas para que sean legibles y descriptivas
        columnas_dummies = [c for c in d.columns if c.startswith('DEPE_')]
        for col_dum in columnas_dummies:
            codigo = col_dum.split('_')[1]
            if codigo in map_depe:
                d.rename(columns={col_dum: map_depe[codigo]}, inplace=True)

    # 3. Escalamiento de variables continuas de retiro (Actividad 3.2)
    columnas_a_escalar = ['RET_HOM_TO', 'RET_MUJ_TO']
    scaler = MinMaxScaler()
    for col in columnas_a_escalar:
        if col in d.columns:
            d[f'{col}_SCALED'] = scaler.fit_transform(d[[col]])

    # 4. Creación de Atributos Derivados (Actividad 3.1)
    # Atributo 1: Retiro total combinado
    if 'RET_HOM_TO' in d.columns and 'RET_MUJ_TO' in d.columns:
        d['RET_TOTAL_TO'] = d['RET_HOM_TO'] + d['RET_MUJ_TO']

    # Atributo 2: Categorización de asistencia según tramos ministeriales
    def categorizar_asistencia(asis):
        if asis < 85.0:
            return '1. Grave (<85%)'
        elif asis < 90.0:
            return '2. Baja (85-90%)'
        elif asis < 95.0:
            return '3. Regular (90-95%)'
        else:
            return '4. Ideal (95-100%)'

    if 'PROM_ASIS' in d.columns:
        d['TRAMO_ASISTENCIA'] = d['PROM_ASIS'].apply(categorizar_asistencia)

    # Registro de métricas finales
    informe['filas_finales'] = len(d)
    return d, informe

# Ejecución del pipeline de limpieza integrado
limpio, informe = limpiar(RUTA + ARCHIVO)
print("Métricas de la Limpieza de Datos:")
print(informe)

Métricas de la Limpieza de Datos:
{'filas_iniciales': 15897, 'filas_finales': 15767}


In [24]:
# Comprobación: ejecuta la función DOS VECES y verifica que da lo mismo.
a, _ = limpiar(RUTA + ARCHIVO)
b, _ = limpiar(RUTA + ARCHIVO)
print('¿Reproducible?', a.equals(b))

¿Reproducible? True


In [25]:
# Guarda el conjunto limpio de forma oficial. Este archivo es el insumo de la semana 3.
if 'limpio' in globals():
    limpio.to_csv(RUTA + 'mi_conjunto_limpio.csv', sep=';', index=False)
    print('✅ ¡Archivo guardado exitosamente!')
    print('Ruta de guardado:', RUTA + 'mi_conjunto_limpio.csv')
    print('Dimensiones finales del archivo:', limpio.shape)
else:
    print('❌ Error: El DataFrame "limpio" no se encuentra en memoria.')

✅ ¡Archivo guardado exitosamente!
Ruta de guardado: mi_conjunto_limpio.csv
Dimensiones finales del archivo: (15767, 224)


---
## Parte 5 · Versionado

Sube el cuaderno y el conjunto limpio a tu repositorio. Un commit con mensaje descriptivo, no «cambios».

In [ ]:
# En una terminal:
# git add S2_Notebook_Laboratorio_CDD.ipynb mi_conjunto_limpio.csv
# git commit -m 'Semana 2: diagnostico de calidad y flujo de limpieza'
# git push


**Enlace a mi repositorio:** _…_

---

## Antes de cerrar el laboratorio

- [ ] Las cinco dimensiones diagnosticadas y registradas en la hoja «Resumen de calidad»
- [ ] Cada transformación aplicada tiene su fila en la bitácora, con justificación
- [ ] Me pronuncié sobre los atípicos: cuáles son error y cuáles fenómeno real
- [ ] Creé al menos un atributo derivado y expliqué para qué sirve
- [ ] La función `limpiar()` corre de principio a fin y entrega lo mismo dos veces
- [ ] El conjunto limpio está guardado y versionado
- [ ] Tengo anotadas las limitaciones que declararé en el informe

Lo que quede pendiente, al Foro de Consultas o a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 2_